# ANAEMIA ANALYSIS USING NATIONAL HEALTH AND NUTRITION EXAMINATION SURVEY (NHANES) DATA.


## 1.0 Introduction

### 1.1 Research Question
Among NHANES participants with anaemia, which observable multidimensional biological patterns are consistent with plausible aetiological mechanisms of multifactorial anaemia, and to what extent do these patterns overlap?

### 1.2 Research Objective
Use routinely collected multidimensional biological data to identify patterns that may distinguish observable, mechanism-related phenotypes of multifactorial anaemia and examine how often or the extent to which these patterns overlap.

#### 1.2.1 Specific Objectives
The goal of this analysis is to:
1. Characterise participants with phenotypes and biological measurements consistent with varying plausible causative mechanisms of complex anaemia.
2. Identify distinct biological patterns among NHANES participants with anaemia using exploratory and unsupervised learning methods.
3. Investigate the extent to which these patterns overlap with one another and align with scientifically established thresholds and biological mechanisms associated with anaemia arising from multiple underlying causes.

### 1.3 Dataset Overview
- Dataset: NHANES data from the 2017-2018 survey cycle.
- Source: The US National Health and Nutrition Examination Survey, conducted by the National Center for Health Statistics (NCHS), a component of the Centers for Disease Control and Prevention (CDC).
- Number of observations: To-be-determined.
- Number of features: To-be-determined.
- Target variable: None for the primary unsupervised analysis.


### 1.4 Feature Description
- In progress

## 2.0 Loading and Initial Inspection

### 2.1 Importing Libraries

In [2]:
import pandas as pd
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import re

### 2.2 Creating a Metadata Table
Loading the datasets and creating a summary file and variable metadata table for each data file.

In [25]:
folder = Path(r"..\data")

  
for file in folder.rglob("*.xpt"):
        
        file_df = pd.read_sas(file)
        
        file_name = file.stem
    
        url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{file_name}.htm"

        response = requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, "html.parser")

        page_header = soup.find("div", id = "PageHeader")
        # print(page_header)

        file_year_header = page_header.find("h2").get_text()
        file_info = {
            "data" : 'info',
            "file_name" : file.stem,
            "file_year" : re.search(r"\d{4}-\d{4}", file_year_header).group(),
            # "source_url" : url,
            "rows_no" : len(file_df),
            "columns_no" : len(file_df.columns),
            "unique_seqn" : file_df["SEQN"].is_unique,
            "number_of_duplicated_seqn" : file_df["SEQN"].duplicated().sum()
                    }
        # print(columns)
        # print(file_year)

        file_table = pd.DataFrame(file_info, index = [0])
        display(file_table.set_index(file_table.columns[0]).T)


        sub_table_info = []
        for column in file_df.columns:
            sub_file_info = {}
            sub_file_info["code_name"] = column

            
            listed_item_text = soup.find("a", href = f"#{column}").get_text().strip()
            # print(listed_item_text)
            # print(repr(listed_item_text))
            listed_item = re.search(r"^(.+?)\s", listed_item_text).group(1)
            # print(listed_item)
            
            actual_name = re.search(r"\w+\s-\s(.+?)(?:\s\([^)]*\))?$", listed_item_text).group(1)

            sub_file_info["actual_name"] = actual_name

            units = re.search(r"^\w+\s-\s(?:.+?)\s(\([^)]*\))$", listed_item_text)
            if units == None:
                units = "none"
            else:
                units = units.group(1)

            # if file_name in ('CBC_J','FETIB_J'): # issue w/ mean cell hgb conc. and transferrin receptor respectively. solved --> .strip()
            #     print(repr(listed_item_text))
            #     print(actual_name)
            #     print(units)

            # print(units)
            sub_file_info["units"] = units
            sub_file_info["obj_type"] = file_df[column].dtypes
            sub_file_info["number_of_missingness"] = file_df[column].isna().sum()
            sub_file_info["percentage_missingness"] = round((file_df[column].isna().mean())*100, 2)

            if column == 'SEQN':
                sub_file_info["max_value"] = 'none'
                sub_file_info["minimum"] = 'none'

            else:
                sub_file_info["max_value"] = file_df[column].max()
                sub_file_info["minimum"] = file_df[column].min()
                # sub_file_info["desc"] = 


            sub_table_info.append(sub_file_info)
        sub_file_table = pd.DataFrame(sub_table_info)
        display(sub_file_table)

data,info
file_name,BIOPRO_J
file_year,2017-2018
rows_no,6401
columns_no,41
unique_seqn,True
number_of_duplicated_seqn,0


,code_name,actual_name,units,obj_type,number_of_missingness,percentage_missingness,max_value,minimum
0,SEQN,Respondent Sequence Number,none,float64,0,0.00,none,none
1,LBXSATSI,Alanine Aminotransferase (ALT),(U/L),float64,499,7.80,420.0,2.0
2,LBDSATLC,ALT Comment Code,none,float64,499,7.80,1.0,0.0
3,LBXSAL,"Albumin, refrigerated serum",(g/dL),float64,496,7.75,5.4,2.1
4,LBDSALSI,"Albumin, refrigerated serum",(g/L),float64,496,7.75,54.0,21.0
5,LBXSAPSI,Alkaline Phosphatase (ALP),(IU/L),float64,498,7.78,638.0,16.0
6,LBXSASSI,Aspartate Aminotransferase (AST),(U/L),float64,519,8.11,272.0,6.0
7,LBXSC3SI,Bicarbonate,(mmol/L),float64,500,7.81,38.0,16.0
8,LBXSBU,Blood Urea Nitrogen,(mg/dL),float64,500,7.81,79.0,2.0
9,LBDSBUSI,Blood Urea Nitrogen,(mmol/L),float64,500,7.81,28.2,0.71


data,info
file_name,CBC_J
file_year,2017-2018
rows_no,8366
columns_no,22
unique_seqn,True
number_of_duplicated_seqn,0


,code_name,actual_name,units,obj_type,number_of_missingness,percentage_missingness,max_value,minimum
0,SEQN,Respondent sequence number,none,float64,0,0.00,none,none
1,LBXWBCSI,White blood cell count,(1000 cells/uL),float64,838,10.02,400.0,1.9
2,LBXLYPCT,Lymphocyte percent,(%),float64,843,10.08,89.7,4.4
3,LBXMOPCT,Monocyte percent,(%),float64,843,10.08,57.2,0.7
4,LBXNEPCT,Segmented neutrophils percent,(%),float64,843,10.08,92.8,8.4
5,LBXEOPCT,Eosinophils percent,(%),float64,843,10.08,29.1,0.0
6,LBXBAPCT,Basophils percent,(%),float64,843,10.08,4.8,0.1
7,LBDLYMNO,Lymphocyte number,(1000 cells/uL),float64,843,10.08,358.8,0.4
8,LBDMONO,Monocyte number,(1000 cells/uL),float64,843,10.08,6.7,0.1
9,LBDNENO,Segmented neutrophils num,(1000 cell/uL),float64,843,10.08,35.2,0.4


data,info
file_name,DEMO_J
file_year,2017-2018
rows_no,9254
columns_no,46
unique_seqn,True
number_of_duplicated_seqn,0


,code_name,actual_name,units,obj_type,number_of_missingness,percentage_missingness,max_value,minimum
0,SEQN,Respondent sequence number,none,float64,0,0.00,none,none
1,SDDSRVYR,Data release cycle,none,float64,0,0.00,10.0,10.0
2,RIDSTATR,Interview/Examination status,none,float64,0,0.00,2.0,1.0
3,RIAGENDR,Gender,none,float64,0,0.00,2.0,1.0
4,RIDAGEYR,Age in years at screening,none,float64,0,0.00,80.0,0.0
5,RIDAGEMN,Age in months at screening - 0 to 24 mos,none,float64,8657,93.55,24.0,0.0
6,RIDRETH1,Race/Hispanic origin,none,float64,0,0.00,5.0,1.0
7,RIDRETH3,Race/Hispanic origin w/ NH Asian,none,float64,0,0.00,7.0,1.0
8,RIDEXMON,Six month time period,none,float64,550,5.94,2.0,1.0
9,RIDEXAGM,Age in months at exam - 0 to 19 years,none,float64,5821,62.90,239.0,0.0


... (6 outputs left out)

data,info
file_name,HSCRP_J
file_year,2017-2018
rows_no,8366
columns_no,3
unique_seqn,True
number_of_duplicated_seqn,0


,code_name,actual_name,units,obj_type,number_of_missingness,percentage_missingness,max_value,minimum
0,SEQN,Respondent Sequence Number,none,float64,0,0.00,none,none
1,LBXHSCRP,HS C-Reactive Protein,(mg/L),float64,1116,13.34,182.82,0.11
2,LBDHRPLC,HS C-Reactive Protein Comment Code,none,float64,1116,13.34,1.0,0.0


data,info
file_name,TFR_J
file_year,2017-2018
rows_no,7332
columns_no,3
unique_seqn,True
number_of_duplicated_seqn,0


,code_name,actual_name,units,obj_type,number_of_missingness,percentage_missingness,max_value,minimum
0,SEQN,Respondent sequence number,none,float64,0,0.00,none,none
1,LBXTFR,Transferrin receptor,(mg/L),float64,765,10.43,41.3,1.01
2,LBDTFRSI,Transferrin receptor,(nmol/L),float64,765,10.43,487.0,11.9


In [4]:
# text = 'LBXMC - Mean Cell Hgb Conc. (g/dL)'

# match = re.search(r"\w+\s-\s(.+?)(?:\s\([^)]*\))?$", text)

# print(match.group(1))

### 2.3 Preview the Datasets

In [ ]:
folder = Path(r"..\data")

for file in folder.glob("*.xpt"):
        file_df = pd.read_sas(file)
        print(file.stem)
        display(file_df.head())
        # display(file_df.tail())

BIOPRO_J


,SEQN,LBXSATSI,LBDSATLC,LBXSAL,LBDSALSI,LBXSAPSI,LBXSASSI,LBXSC3SI,LBXSBU,LBDSBUSI,...,LBXSCA,LBDSCASI,LBXSCH,LBDSCHSI,LBXSTP,LBDSTPSI,LBXSTR,LBDSTRSI,LBXSUA,LBDSUASI
0,93705.0,16.0,5.397605e-79,4.4,44.0,74.0,20.0,31.0,11.0,3.93,...,9.2,2.300,157.0,4.060,7.3,73.0,95.0,1.073,5.8,345.0
1,93706.0,10.0,5.397605e-79,4.4,44.0,79.0,14.0,28.0,12.0,4.28,...,9.6,2.400,149.0,3.853,7.1,71.0,92.0,1.039,8.0,475.8
2,93707.0,13.0,5.397605e-79,5.2,52.0,238.0,24.0,22.0,17.0,6.07,...,10.1,2.525,199.0,5.146,8.0,80.0,110.0,1.242,5.5,327.1
3,93708.0,19.0,5.397605e-79,3.9,39.0,66.0,21.0,27.0,16.0,5.71,...,9.5,2.375,210.0,5.431,7.1,71.0,72.0,0.813,4.5,267.7
4,93709.0,15.0,5.397605e-79,3.7,37.0,86.0,17.0,24.0,20.0,7.14,...,9.9,2.475,180.0,4.655,7.0,70.0,132.0,1.490,6.2,368.8


,SEQN,LBXSATSI,LBDSATLC,LBXSAL,LBDSALSI,LBXSAPSI,LBXSASSI,LBXSC3SI,LBXSBU,LBDSBUSI,...,LBXSCA,LBDSCASI,LBXSCH,LBDSCHSI,LBXSTP,LBDSTPSI,LBXSTR,LBDSTRSI,LBXSUA,LBDSUASI
6396,102952.0,22.0,5.397605e-79,4.8,48.0,57.0,27.0,27.0,13.0,4.64,...,9.9,2.475,121.0,3.129,7.4,74.0,98.0,1.106,6.4,380.7
6397,102953.0,40.0,5.397605e-79,4.0,40.0,115.0,29.0,24.0,17.0,6.07,...,9.0,2.250,184.0,4.758,7.5,75.0,114.0,1.287,5.8,345.0
6398,102954.0,6.0,5.397605e-79,3.9,39.0,55.0,15.0,21.0,8.0,2.86,...,9.0,2.250,172.0,4.448,6.5,65.0,64.0,0.723,3.1,184.4
6399,102955.0,11.0,5.397605e-79,4.1,41.0,84.0,16.0,23.0,11.0,3.93,...,9.2,2.300,151.0,3.905,7.5,75.0,89.0,1.005,6.3,374.7
6400,102956.0,47.0,5.397605e-79,4.3,43.0,84.0,27.0,27.0,12.0,4.28,...,9.7,2.425,167.0,4.319,7.2,72.0,289.0,3.263,6.4,380.7


CBC_J


,SEQN,LBXWBCSI,LBXLYPCT,LBXMOPCT,LBXNEPCT,LBXEOPCT,LBXBAPCT,LBDLYMNO,LBDMONO,LBDNENO,...,LBXRBCSI,LBXHGB,LBXHCT,LBXMCVSI,LBXMCHSI,LBXMC,LBXRDW,LBXPLTSI,LBXMPSI,LBXNRBC
0,93703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,7.4,47.8,8.0,42.6,1.0,0.7,3.5,0.6,3.2,...,4.25,13.1,37.0,87.0,30.8,35.4,12.8,239.0,8.6,1.000000e-01
2,93705.0,8.6,40.0,7.4,48.8,2.9,1.0,3.4,0.6,4.2,...,5.48,11.9,36.7,67.0,21.7,32.4,15.6,309.0,7.9,5.397605e-79
3,93706.0,6.1,24.6,9.1,61.4,4.3,0.8,1.5,0.6,3.7,...,5.24,16.3,47.0,89.7,31.1,34.7,12.2,233.0,6.6,5.397605e-79
4,93707.0,11.2,37.1,6.2,54.7,1.6,0.5,4.2,0.7,6.1,...,5.02,14.5,42.1,83.9,28.9,34.4,13.6,348.0,8.5,2.000000e-01


,SEQN,LBXWBCSI,LBXLYPCT,LBXMOPCT,LBXNEPCT,LBXEOPCT,LBXBAPCT,LBDLYMNO,LBDMONO,LBDNENO,...,LBXRBCSI,LBXHGB,LBXHCT,LBXMCVSI,LBXMCHSI,LBXMC,LBXRDW,LBXPLTSI,LBXMPSI,LBXNRBC
8361,102952.0,5.1,34.9,7.1,56.2,1.3,0.6,1.8,0.4,2.9,...,4.64,13.4,40.6,87.7,29.0,33.1,13.3,161.0,9.1,5.000000e-01
8362,102953.0,8.3,28.3,6.8,60.8,3.6,0.6,2.3,0.6,5.0,...,5.01,15.5,44.4,88.8,30.9,34.8,13.6,227.0,8.7,1.000000e-01
8363,102954.0,5.1,43.5,11.8,39.9,3.5,1.4,2.2,0.6,2.0,...,3.72,6.4,23.1,61.9,17.3,27.9,19.6,145.0,9.7,1.000000e-01
8364,102955.0,11.4,33.4,7.3,57.5,1.1,0.9,3.8,0.8,6.6,...,4.93,11.4,33.7,68.4,23.1,33.7,17.4,382.0,7.7,1.000000e-01
8365,102956.0,9.0,29.4,6.9,58.8,4.0,1.0,2.6,0.6,5.3,...,5.46,17.4,50.1,91.8,31.9,34.7,13.3,159.0,8.5,5.397605e-79


... (14 outputs left out)

,SEQN,LBXHSCRP,LBDHRPLC
8361,102952.0,0.45,5.397605e-79
8362,102953.0,1.84,5.397605e-79
8363,102954.0,0.81,5.397605e-79
8364,102955.0,5.11,5.397605e-79
8365,102956.0,3.70,5.397605e-79


TFR_J


,SEQN,LBXTFR,LBDTFRSI
0,93703.0,NaN,NaN
1,93704.0,2.59,30.6
2,93705.0,3.37,39.8
3,93706.0,2.88,34.0
4,93707.0,2.88,34.0


,SEQN,LBXTFR,LBDTFRSI
7327,102952.0,2.47,29.1
7328,102953.0,2.42,28.6
7329,102954.0,21.00,248.0
7330,102955.0,4.89,57.7
7331,102956.0,1.98,23.4


## 3.0 Data Preparation